# Connector: schedulers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/connector_scheduler.ipynb)

Policies such as `train_every: 7d` and `revalidate: {every: 365d}` are evaluated by `pecca scheduler run`. A `Scheduler` connector registers that command with cron, Airflow or Databricks Workflows.

In [1]:
%pip install -q pecca

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


In [3]:
from pecca import connectors
cron = connectors.create("scheduler", {"type": "cron", "path": "jobs.json"})
cron.register_job("pecca-tick", "1d", ["pecca", "scheduler", "run", "--once"])
cron.register_job("pecca-weekly", "7d", ["pecca", "scheduler", "run", "--once"])
cron.list_jobs()

# pecca job pecca-tick
0 6 * * * pecca scheduler run --once
# pecca job pecca-weekly
0 6 */7 * * pecca scheduler run --once


[{'name': 'pecca-tick',
  'every': '1d',
  'command': ['pecca', 'scheduler', 'run', '--once'],
  'crontab': '0 6 * * * pecca scheduler run --once'},
 {'name': 'pecca-weekly',
  'every': '7d',
  'command': ['pecca', 'scheduler', 'run', '--once'],
  'crontab': '0 6 */7 * * pecca scheduler run --once'}]

## Make a call due and run one tick

In [4]:
import yaml
logs = generate(3000)
logs.to_csv("logs.csv", index=False)                       # the call's history: a retrain reads it from the project's datasource
Path("pecca.yaml").write_text(yaml.safe_dump({"version": 1, "projects": {"default": {
    "datasource": {"type": "csv", "path": "logs.csv", "columns": {"input": {"subject": "email_subject", "body": "email_body"}, "llm_output": "llm_rfi",
                                                              "human_label": "human_rfi", "call_name": {"literal": "route_rfi"}},
                   "input_template": "{subject}\n\n{body}"},
    "governance": {"extends": "default"}, "calls": {"route_rfi": {"policy": {"train_every": "7d", "latency_budget_ms": 5}}}}}}))   # a 5 ms budget keeps scheduled retrains to the fast models
context.reset()
print(pecca.train("route_rfi", candidates=["tfidf_linear"]).version)
from pecca.governance import schedule
from pecca.core.timeutil import iso, now
from datetime import timedelta
call = context.get_call("route_rfi")
print("next dates:", {k: v.date().isoformat() for k, v in schedule.next_dates(call).items()})
st = call.state(); st.last_trained = iso(now() - timedelta(days=8)); call.save_state(st)     # pretend it was trained 8 days ago
print("due now:", schedule.due_items(call, now()))

v1
next dates: {'retrain': '2026-10-13', 'revalidate': '2027-10-06', 'drift_review': '2026-11-05'}
due now: ['retrain']


In [5]:
!pecca scheduler run --once

default/default/route_rfi: retrained


In [6]:
print("versions after the tick:", [m.version for m in context.get_call("route_rfi").versions()])

versions after the tick: ['v1', 'v2']


## Airflow and Databricks Workflows
The Airflow connector writes a DAG file per job (and exposes a lazy `PeccaOperator`); Databricks Workflows writes job JSON.

In [7]:
af = connectors.create("scheduler", {"type": "airflow", "dags_dir": "dags"})
af.register_job("route_rfi", "7d", ["pecca", "scheduler", "run", "--once"])
print(Path("dags/pecca_route_rfi.py").read_text())
dbx = connectors.create("scheduler", {"type": "databricks_workflows", "out_dir": "dbx_jobs", "cluster_id": "0000-demo-cluster"})
dbx.register_job("route_rfi", "7d", ["pecca", "scheduler", "run", "--once"])
print(json.dumps(dbx.list_jobs()[0], indent=2))

"""Generated by Pecca. Edit freely."""
from datetime import datetime

from airflow import DAG
from airflow.operators.bash import BashOperator

with DAG("pecca_route_rfi", schedule="0 6 */7 * *", start_date=datetime(2026, 1, 1), catchup=False) as dag:
    BashOperator(task_id="route_rfi", bash_command="pecca scheduler run --once")

{
  "name": "pecca-route_rfi",
  "tasks": [
    {
      "task_key": "route_rfi",
      "spark_python_task": {
        "python_file": "pecca_job.py",
        "parameters": [
          "scheduler",
          "run",
          "--once"
        ]
      },
      "existing_cluster_id": "0000-demo-cluster"
    }
  ],
  "schedule": {
    "quartz_cron_expression": "0 0 6 ? * *",
    "timezone_id": "UTC"
  }
}
